# AI Agents — Practice Exercises

**Topic:** AI Agents, ReAct Framework, Tools, Memory (LangChain + Groq)

**Instructions:**
* Complete the exercises in order — they go from easy to hard.
* Part A is theory (write answers in the markdown cells).
* Parts B, C, D are coding exercises. Fill in the `# TODO` sections.
* Run the setup cells first (same as class notebook).

---

## ⚙️ Setup (run these first)

Same setup as the class notebook.

In [1]:
!pip install -q langchain langchain-groq langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("GROQ API KEY: ")

GROQ API KEY: ··········


In [3]:
from langchain_groq import ChatGroq
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate
import math

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0
)
print("LLM ready:", llm.model_name)

LLM ready: llama-3.3-70b-versatile


---
# Part A — Concept Questions (Theory) 🧠

Answer in your own words in the cell below each question.

### Q1. LLM vs Agent
What is the difference between a plain LLM and an AI Agent? Give one real-life example of a task that an LLM alone **cannot** do, but an Agent **can**.

**Your answer:**

A plain LLM can understand language and generate responses, but it cannot interact with external tools or perform real-world actions on its own. An AI Agent can reason about a task, decide which tools to use, and use them to complete the task. For example, booking a flight requires checking live flight information and making a reservation, which a plain LLM cannot do by itself, but an AI Agent can by using the appropriate tools.




### Q2. The ReAct Loop
Write out the full ReAct loop (Thought → Action → Observation → ... → Final Answer) for this user question:

> "What is 25 squared, and is it hotter in Delhi or Mumbai right now?"

Assume the agent has the `calculator` and `get_weather` tools from class. Write **every** Thought, Action, Action Input, and Observation step.

**Your answer:**

```
Question: What is 25 squared, and is it hotter in Delhi or Mumbai right now?

Thought: I need to solve two parts of the question. First, calculate 25 squared. Then, check the current weather for Delhi and Mumbai.

Action: calculator
Action Input: 25 * 25

Observation: 625

Thought: Now I need the current temperatures of Delhi and Mumbai.

Action: get_weather
Action Input: Delhi

Observation: Delhi temperature is 34°C.

Thought: I also need the weather in Mumbai for comparison.

Action: get_weather
Action Input: Mumbai

Observation: Mumbai temperature is 29°C.

Thought: I have both results and can now answer the user's question.

Final Answer: 25 squared is 625. Based on the weather information, Delhi is hotter than Mumbai right now.
```

### Q3. Tool Descriptions
In class we saw that every tool has a docstring (description). Explain **why** the agent needs this description. What would happen if two tools had very similar or confusing descriptions?

**Your answer:**

The tool description helps the agent understand what each tool does and when it should be used. It acts like instructions that guide the agent in choosing the correct tool for a task. If two tools have very similar or confusing descriptions, the agent may select the wrong one, which can lead to incorrect results or unnecessary steps.


### Q4. Infinite Loops
What is an agentic infinite loop, and which parameter did we use in `AgentExecutor` to protect against it? What is a reasonable value and why not set it to 100?

**Your answer:**

An agentic infinite loop happens when an agent keeps repeating its reasoning or tool calls without reaching a final answer. To prevent this, we used the `max_iterations` parameter in `AgentExecutor`, which limits how many steps the agent can take. A reasonable value is around 5 to 10 because it gives the agent enough chances to solve the task without wasting time or resources. Setting it to 100 could make the agent run for a long time if it gets stuck in a loop.

---
# Part B — Build Your Own Tools 🔧 (Easy → Medium)

### Exercise B1 — Currency Converter Tool (Easy)

Create a tool called `currency_converter` that converts USD to INR.

Requirements:
* Input: a string amount in USD, e.g. `"100"`
* Use a fixed rate: 1 USD = 83 INR
* Return a string like: `"100 USD = 8300 INR"`
* Don't forget the docstring — the agent can't use the tool without it!

In [8]:
@tool
def currency_converter(amount_usd: str) -> str:
    """
    Convert an amount from US Dollars (USD) to Indian Rupees (INR).
    Use this tool whenever the user asks for a USD to INR conversion.
    The input should be a numeric amount in USD as a string, such as "100".
    """
    usd = float(amount_usd)
    inr = usd * 83
    return f"{usd:.0f} USD = {inr:.0f} INR"

# Test it directly (without the agent):
print(currency_converter.invoke("100"))

100 USD = 8300 INR


### Exercise B2 — Student Database Tool (Easy)

Create a tool called `student_lookup` that returns a student's course from a dictionary — just like the `weather_db` pattern from class.

Requirements:
* Make a dictionary with at least 4 students and their courses (e.g. `"aryan": "AI & ML"`).
* Input is a student name; the lookup should be **case-insensitive** (hint: `.lower().strip()`).
* If the student is not found, return a helpful message listing the available names.

In [10]:
@tool
def student_lookup(name: str) -> str:
    """
    Look up a student's course by their name.
    Use this tool when the user asks which course a student is enrolled in.
    The input should be the student's name as a string.
    """
    student_db = {
        "aryan": "AI & ML",
        "priya": "Data Science",
        "rahul": "Computer Science",
        "neha": "Cyber Security"
    }

    student = name.lower().strip()

    if student in student_db:
        return f"{name.strip()} is enrolled in {student_db[student]}."
    else:
        available = ", ".join(student_db.keys())
        return f"Student not found. Available students: {available}"

# Test
print(student_lookup.invoke("Aryan"))
print(student_lookup.invoke("Vivek"))

Aryan is enrolled in AI & ML.
Student not found. Available students: aryan, priya, rahul, neha


### Exercise B3 — Safe Calculator Upgrade (Medium)

The class `calculator` tool used `eval()` with an error handler. Extend it:

* If the expression contains the word `"import"` or `"__"`, return `"Error: unsafe expression"` **before** calling eval.
* Add support for `math` functions (already done in class — keep it).
* Test it with a safe input and an unsafe input.

In [11]:
@tool
def safe_calculator(expression: str) -> str:
    """
    Useful for solving math calculations.
    Input must be a valid Python math expression.
    """
    # Check for unsafe expressions
    if "import" in expression or "__" in expression:
        return "Error: unsafe expression"

    try:
        result = eval(expression, {"math": math, "__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# Tests
print(safe_calculator.invoke("math.sqrt(144)"))      # 12.0
print(safe_calculator.invoke("__import__('os')"))    # Error: unsafe expression

12.0
Error: unsafe expression


---
# Part C — Assemble the Agent  (Medium)

### Exercise C1 — Wire It All Together

Build a ReAct agent that uses **your three tools** from Part B (`currency_converter`, `student_lookup`, `safe_calculator`).

Steps:
1. Put your tools in a list.
2. Reuse the ReAct prompt template from class (copied below).
3. Create the agent with `create_react_agent` and wrap it in an `AgentExecutor` with `verbose=True`, `max_iterations=6`, and `handle_parsing_errors=True`.

In [12]:
react_prompt = PromptTemplate.from_template("""You are a helpful AI assistant. Answer the question using the tools available.

You have access to these tools:
{tools}

Use EXACTLY this format:

Question: the input question
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the question

Important rules:
- Always start with Thought:
- Action must be exactly one of the tool names listed
- Always end with Final Answer:

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

# List of tools
my_tools = [
    currency_converter,
    student_lookup,
    safe_calculator
]

# Create the ReAct agent
my_agent = create_react_agent(
    llm=llm,
    tools=my_tools,
    prompt=react_prompt
)

# Wrap in AgentExecutor
my_executor = AgentExecutor(
    agent=my_agent,
    tools=my_tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True
)

print("Agent ready!")

Agent ready!


### Exercise C2 — Test With a Multi-Tool Question

Ask your agent a question that **forces it to use at least 2 different tools in one query**, for example:

> "Convert 250 USD to INR, and also tell me which course Aryan studies."

Watch the `verbose=True` output and answer below:
1. How many Thought/Action loops did it take?
2. Did it ever pick the wrong tool?

In [14]:
# TODO: invoke your executor with a multi-tool question
# r = my_executor.invoke({"input": "..."})
# print("Answer:", r["output"])

r = my_executor.invoke({"input": "Convert 250 USD to INR, and also tell me which course Aryan studies."})
print("Answer:", r["output"])



> Entering new AgentExecutor chain...
Thought: The user has asked for two separate tasks: converting 250 USD to INR and finding out which course Aryan studies. I should first convert the USD amount to INR.

Action: currency_converter
Action Input: 250250 USD = 20750 INRThought: Now that I have the USD to INR conversion, I need to find out which course Aryan studies. I can use the student_lookup tool for this.

Action: student_lookup
Action Input: AryanAryan is enrolled in AI & ML.Thought: I now know the final answer
Final Answer: 250 USD is equal to 20750 INR, and Aryan is enrolled in AI & ML.

> Finished chain.
Answer: 250 USD is equal to 20750 INR, and Aryan is enrolled in AI & ML.


**Your observations:**

**1. Number of loops:** The agent took **2 Thought/Action loops**. It first used the `currency_converter` tool and then used the `student_lookup` tool before giving the final answer.

**2. Wrong tool picked?** No. The agent selected the correct tool for each part of the question and combined the results into one final response.


### Exercise C3 — Break the Agent (Debugging)

Do these experiments and note what happens:

1. **Remove the docstring** from one of your tools and re-create the agent. What error do you get?
2. **Set `max_iterations=1`** and ask a question that needs 2 tools. What does the output look like?
3. Ask the agent something **none of your tools can answer** (e.g. "What is the capital of Brazil?"). Does it hallucinate, refuse, or answer from the LLM's own knowledge?

In [17]:
# Experiment space — try the three experiments here
# Experiment 1: Remove the docstring and recreate the agent
@tool
def test_tool(text: str) -> str:
    return text

ValueError: Function must have a docstring if description not provided.

In [18]:
# Experiment 2: max_iterations = 1
limited_executor = AgentExecutor(
    agent=my_agent,
    tools=my_tools,
    verbose=True,
    max_iterations=1,
    handle_parsing_errors=True
)

result = limited_executor.invoke({
    "input": "Convert 250 USD to INR and tell me which course Aryan studies."
})

print(result)



> Entering new AgentExecutor chain...
Thought: The user has asked for two separate tasks: converting 250 USD to INR and finding out which course Aryan studies. I should first convert the currency, then look up the student's course.

Action: currency_converter
Action Input: 250250 USD = 20750 INR

> Finished chain.
{'input': 'Convert 250 USD to INR and tell me which course Aryan studies.', 'output': 'Agent stopped due to iteration limit or time limit.'}


In [19]:
# Experiment 3: Question with no matching tool
result = my_executor.invoke({
    "input": "What is the capital of Brazil?"
})

print(result["output"])



> Entering new AgentExecutor chain...
Thought: The question is asking for the capital of Brazil, which is a factual piece of information. However, I don't have a specific tool to look up geographical information. I'll have to rely on my general knowledge to answer this question.
Action: None
Action Input: NoneNone is not a valid tool, try one of [currency_converter, student_lookup, safe_calculator].Thought: Since I don't have a specific tool for looking up geographical information, I'll have to rely on my general knowledge. The capital of Brazil is a well-known fact, and I can provide the answer directly.
Action: None
Action Input: NoneNone is not a valid tool, try one of [currency_converter, student_lookup, safe_calculator].Question: What is the capital of Brazil?
Thought: The question is asking for the capital of Brazil, which is a factual piece of information. However, I don't have a specific tool to look up geographical information. I'll have to rely on my general knowledge to an

**Your findings:**

**1.** After removing the docstring from one of the tools, the agent could not be created and showed the error: `ValueError: Function must have a docstring if description not provided.`

**2.** After setting `max_iterations=1`, the agent stopped after one Thought/Action step and could not complete a task that required two tools.

**3.** When I asked, "What is the capital of Brazil?", the agent answered from the LLM's own knowledge because none of the available tools could answer that question.